# Fri and Merkle Tree

## Merkle Trees

In [2]:
from hashlib import sha3_256
from zk_adventures_types import F

x = F(3)
hash = sha3_256(x.to_bytes()).hexdigest()
print(hash)

y = sha3_256(bytes.fromhex(hash)).hexdigest()
print(y)

bae484b67d98e56926e87f8cd01da5f32d6916125cc0b04bc9b6ebbe02f950b5
baafe263a6de91cf278fb031449742aa6dfbee51e62f65d94c84204fe2195e12


In [3]:
V = [F(i) for i in range(1, 17)]


In [29]:
import math

def merkle_commitment(leaves):
    merkle_tree = []
    if len(leaves) == 1:
        return sha3_256(leaves[0].to_bytes()).hexdigest()
    else:

        leaves_as_bytes = [leaf.to_bytes() for leaf in leaves]
        current_level = [sha3_256(leaf).hexdigest() for leaf in leaves_as_bytes]
        merkle_tree.append(current_level)
        
        for _ in range(0, int(math.log2(len(leaves)))):
            next_merkle_level = []

            for i in range(0, len(current_level)-1, 2):
                left = current_level[i]
                right = current_level[i + 1]
                
                combined = left.encode() + right.encode()
                parent_hash = sha3_256(combined).hexdigest()

                next_merkle_level.append(parent_hash)
            merkle_tree.append(next_merkle_level)
            current_level = next_merkle_level
    return merkle_tree

def merkle_open(merkle_tree, index):
    proof = []
    for level in merkle_tree[0:len(merkle_tree)-1]:

        if index % 2 == 0:
            sibling_index = index + 1
        else:
            sibling_index = index - 1
        
        proof.append(level[sibling_index])
        index //= 2
    return proof

cm = merkle_commitment(V)
for level in cm:
    print(level)

print(merkle_open(cm,3))

['c071a92a8fd2403faf107a10ea2b2524296241894869d96a77031e3a804721e4', 'a50457e609c84942d4ce6fa24c8e0588cb274d594505c16f939f060de9970d44', 'bae484b67d98e56926e87f8cd01da5f32d6916125cc0b04bc9b6ebbe02f950b5', '43c0b46b8c8c23ac96b093915ff0850808076578e2cbf3a1b8262eb7115d4ac8', '3e6bd262e415f795f73328bb63905e69d3b25a3706c3019ef548913d1117dcbd', 'df5a88e34e683b0bfe79b3fdbfff8d636a1f437a4cae8fa1ceecc1c3b544117d', '0fe7a208dfe7d8d74b9b7d080c198f0cd0546a49b629700c85881cda8c3f7e1c', '0c4da8becd5cf9e35009f7cd50a31772815def24a2ca90abeae05680f4d3b991', '4a9efb55a0fdd5e600730ce3b0d0cc60d37e93dd50f2dfdeaa19907afa3a38e6', 'b8a5b43be0db37637101a9d51d45a5fd8668fd71f5fe40643dbd1cfd927f0600', 'f30a29520f5525ea9e06b079ba4e2def88e3c832f0caf5a47c1ebcfb1e1ae873', '757e6d122e513e8d7975b2389fdda7da7a4bc7d752c629d5f51a25d81b2626bf', 'c94e5963cb08e70345d405717d3c967b05c3503dd4945719e97f82bcdca66bed', 'e934f503d3303ca955b92d1e3b42aa624be6f220e5ae3dec8bc73d6937bee5a8', '4f5d9972735d30300afe141eb6532ae62916c7af77e2f4

In [ ]:
def verify_merkle_proof(leaf, proof, root, index):
    current_hash = sha3_256(leaf.to_bytes()).hexdigest()
    for sibling_hash in proof:
        if index % 2 == 0:
            combined = current_hash.encode() + sibling_hash.encode()
        else:
            combined = sibling_hash.encode() + current_hash.encode()
        
        current_hash = sha3_256(combined).hexdigest()
        index //= 2
    return current_hash == root

root = cm[-1][0]
verify_merkle_proof(V[3], merkle_open(cm, 3), cm[-1][0], 3)


4
43c0b46b8c8c23ac96b093915ff0850808076578e2cbf3a1b8262eb7115d4ac8
291d7317a5f39b160cb53bf7563f6300ddf51c35bde8a741dcdb2e05ba3dbff6
291d7317a5f39b160cb53bf7563f6300ddf51c35bde8a741dcdb2e05ba3dbff6


True

In [37]:
V = [F(i) for i in range(1, 1025)]
cm = merkle_commitment(V)
index = 458
proof = merkle_open(cm, index)
root = cm[-1][0]
print(verify_merkle_proof(V[index], proof, root, index))

459
f3abf600cba4dd8cf9a40157fa094b6bb2fedf2da675652fe9760f3fd5eecdf3
c21d66f55fba64803b4c4f05d327aeb95153eb22408d9fc734d0e72841f76a28
c21d66f55fba64803b4c4f05d327aeb95153eb22408d9fc734d0e72841f76a28
True


## Fri


### Building the domains


In [ ]:
# generator 
# omega generator of a cyclic group

#build the cosets of the powers of omega with g

### Folding Phase

In [ ]:
# commit the polynomial f by evaluating it in the corresponding domain and building a merkle tree on top of it

# the verifier sends challenges to the prover, which they use to repetedly fold the polynomial
# the prover sends the final folded polynomial, which is just a constant.

### Query phase

In [ ]:
# The verifier asks the prover to open the commitments at random points of the domains and checks if the equality holds.